In [10]:
import cv2
import numpy as np
import pyautogui
from PIL import ImageGrab
import screeninfo
from typing import List, Dict, Optional, Tuple, Union
import time, random

In [11]:
class MultiMonitorSquareDetector:
    def __init__(self):
        self.monitors = self.get_monitor_info()
        self.primary_monitor = self.get_primary_monitor()
    
    def get_monitor_info(self) -> List[Dict]:
        """获取所有显示器信息"""
        monitors = []
        try:
            for i, monitor in enumerate(screeninfo.get_monitors()):
                monitors.append({
                    'index': i,
                    'name': monitor.name if hasattr(monitor, 'name') else f"Monitor {i+1}",
                    'x': monitor.x,
                    'y': monitor.y,
                    'width': monitor.width,
                    'height': monitor.height,
                    'is_primary': monitor.is_primary if hasattr(monitor, 'is_primary') else (i == 0)
                })
        except Exception as e:
            print(f"警告：无法获取显示器信息，使用默认配置: {e}")
            # 如果screeninfo失败，使用pyautogui获取主显示器信息
            screen_size = pyautogui.size()
            monitors.append({
                'index': 0,
                'name': "Primary Monitor",
                'x': 0,
                'y': 0,
                'width': screen_size.width,
                'height': screen_size.height,
                'is_primary': True
            })
        return monitors
    
    def get_primary_monitor(self):
        """获取主显示器信息"""
        for monitor in self.monitors:
            if monitor['is_primary']:
                return monitor
        return self.monitors[0] if self.monitors else None
    
    def list_monitors(self) -> None:
        """列出所有显示器信息"""
        print("可用显示器:")
        for monitor in self.monitors:
            primary_tag = " [主显示器]" if monitor['is_primary'] else ""
            print(f"  {monitor['index']}: {monitor['name']}{primary_tag}")
            print(f"    位置: ({monitor['x']}, {monitor['y']})")
            print(f"    分辨率: {monitor['width']} x {monitor['height']}")
            print()
    
    def capture_monitor(self, monitor_index: Optional[int] = None) -> np.ndarray:
        """
        截取指定显示器的屏幕
        monitor_index: 显示器索引，None表示截取所有显示器
        """
        if monitor_index is None:
            # 截取所有显示器
            screenshot = ImageGrab.grab()
        else:
            if monitor_index < 0 or monitor_index >= len(self.monitors):
                raise ValueError(f"显示器索引 {monitor_index} 无效，可用范围: 0-{len(self.monitors)-1}")
            
            monitor = self.monitors[monitor_index]
            # 截取指定显示器区域
            bbox = (
                monitor['x'], 
                monitor['y'], 
                monitor['x'] + monitor['width'], 
                monitor['y'] + monitor['height']
            )
            screenshot = ImageGrab.grab(bbox)
        
        return cv2.cvtColor(np.array(screenshot), cv2.COLOR_RGB2BGR)
    
    def find_colored_squares(self, 
                           color_bgr: Tuple[int, int, int],
                           min_area: int = 100,
                           max_area: int = 10000,
                           monitor_index: Optional[int] = None,
                           color_tolerance: int = 10,
                           aspect_ratio_tolerance: float = 0.1) -> List[Dict]:
        """
        在指定显示器上寻找指定颜色的正方形
        
        Args:
            color_bgr: BGR格式的颜色值
            min_area: 最小面积
            max_area: 最大面积
            monitor_index: 显示器索引，None表示搜索所有显示器
            color_tolerance: 颜色容差
            aspect_ratio_tolerance: 长宽比容差
        
        Returns:
            包含正方形信息的字典列表
        """
        if monitor_index is not None:
            return self._find_squares_single_monitor(
                color_bgr, min_area, max_area, monitor_index, 
                color_tolerance, aspect_ratio_tolerance
            )
        else:
            # 在所有显示器上搜索
            all_squares = []
            for i in range(len(self.monitors)):
                squares = self._find_squares_single_monitor(
                    color_bgr, min_area, max_area, i, 
                    color_tolerance, aspect_ratio_tolerance
                )
                all_squares.extend(squares)
            return all_squares
    
    def _find_squares_single_monitor(self, 
                                   color_bgr: Tuple[int, int, int],
                                   min_area: int,
                                   max_area: int,
                                   monitor_index: int,
                                   color_tolerance: int,
                                   aspect_ratio_tolerance: float) -> List[Dict]:
        """在单个显示器上寻找正方形"""
        
        if monitor_index < 0 or monitor_index >= len(self.monitors):
            raise ValueError(f"显示器索引 {monitor_index} 无效")
        
        monitor = self.monitors[monitor_index]
        screenshot = self.capture_monitor(monitor_index)
        
        # 创建颜色掩码
        lower_bound = np.array([max(0, c - color_tolerance) for c in color_bgr])
        upper_bound = np.array([min(255, c + color_tolerance) for c in color_bgr])
        mask = cv2.inRange(screenshot, lower_bound, upper_bound)
        
        # 形态学操作，清理噪声
        kernel = np.ones((3, 3), np.uint8)
        mask = cv2.morphologyEx(mask, cv2.MORPH_CLOSE, kernel)
        mask = cv2.morphologyEx(mask, cv2.MORPH_OPEN, kernel)
        
        # 查找轮廓
        contours, _ = cv2.findContours(mask, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)
        
        squares = []
        for contour in contours:
            area = cv2.contourArea(contour)
            if min_area <= area <= max_area:
                # 近似轮廓为多边形
                epsilon = 0.02 * cv2.arcLength(contour, True)
                approx = cv2.approxPolyDP(contour, epsilon, True)
                
                # 检查是否为4边形
                if len(approx) == 4:
                    # 获取边界框
                    x, y, w, h = cv2.boundingRect(contour)
                    
                    # 检查长宽比
                    aspect_ratio = float(w) / h
                    if abs(aspect_ratio - 1.0) <= aspect_ratio_tolerance:
                        # 转换为全局坐标
                        global_x = monitor['x'] + x
                        global_y = monitor['y'] + y
                        global_center_x = monitor['x'] + x + w // 2
                        global_center_y = monitor['y'] + y + h // 2
                        
                        squares.append({
                            'monitor_index': monitor_index,
                            'monitor_name': monitor['name'],
                            'center': (global_center_x, global_center_y),
                            'center_relative': (x + w//2, y + h//2),  # 相对于显示器的坐标
                            'top_left': (global_x, global_y),
                            'top_left_relative': (x, y),  # 相对于显示器的坐标
                            'width': w,
                            'height': h,
                            'area': area,
                            'aspect_ratio': aspect_ratio
                        })
        
        return squares
    
    def find_squares_by_size(self,
                           color_bgr: Tuple[int, int, int],
                           target_size: int,
                           monitor_index: Optional[int] = None,
                           size_tolerance: int = 5,
                           color_tolerance: int = 10) -> List[Dict]:
        """
        根据指定大小寻找正方形
        
        Args:
            color_bgr: BGR格式的颜色值
            target_size: 目标边长
            monitor_index: 显示器索引
            size_tolerance: 大小容差
            color_tolerance: 颜色容差
        """
        min_area = (target_size - size_tolerance) ** 2
        max_area = (target_size + size_tolerance) ** 2
        
        return self.find_colored_squares(
            color_bgr, min_area, max_area, monitor_index, color_tolerance
        )
    
    def visualize_results(self, squares: List[Dict], save_path: Optional[str] = None) -> None:
        """
        可视化检测结果
        
        Args:
            squares: 检测到的正方形列表
            save_path: 保存路径，None表示不保存
        """
        if not squares:
            print("没有找到正方形")
            return
        
        # 按显示器分组
        monitor_squares = {}
        for square in squares:
            monitor_idx = square['monitor_index']
            if monitor_idx not in monitor_squares:
                monitor_squares[monitor_idx] = []
            monitor_squares[monitor_idx].append(square)
        
        # 为每个有检测结果的显示器创建可视化
        for monitor_idx, monitor_square_list in monitor_squares.items():
            screenshot = self.capture_monitor(monitor_idx)
            
            # 在图像上标记正方形
            for i, square in enumerate(monitor_square_list):
                x, y = square['top_left_relative']
                w, h = square['width'], square['height']
                
                # 绘制边界框
                cv2.rectangle(screenshot, (x, y), (x + w, y + h), (0, 255, 0), 2)
                
                # 标记中心点
                center = square['center_relative']
                cv2.circle(screenshot, center, 5, (0, 0, 255), -1)
                
                # 添加文本标签
                label = f"#{i+1} ({w}x{h})"
                cv2.putText(screenshot, label, (x, y - 10), 
                          cv2.FONT_HERSHEY_SIMPLEX, 0.6, (255, 255, 255), 2)
            
            # 显示结果
            window_name = f"Monitor {monitor_idx} - Found {len(monitor_square_list)} squares"
            cv2.imshow(window_name, screenshot)
            
            # 保存结果
            if save_path:
                save_filename = f"{save_path}_monitor_{monitor_idx}.png"
                cv2.imwrite(save_filename, screenshot)
                print(f"结果已保存到: {save_filename}")
        
        print("按任意键关闭窗口...")
        cv2.waitKey(0)
        cv2.destroyAllWindows()


In [4]:
# 使用示例
def main():
    detector = MultiMonitorSquareDetector()
    
    # 列出所有显示器
    detector.list_monitors()
    
    # 在所有显示器上寻找红色正方形
    print("在所有显示器上寻找正方形...")
    red_squares = detector.find_colored_squares(
        color_bgr=(123, 123, 123),  # 
        min_area=100,
        max_area=200
    )
    
    print(f"找到 {len(red_squares)} 个正方形:")
    for i, square in enumerate(red_squares):
        print(f"正方形 {i+1}:")
        print(f"  显示器: {square['monitor_name']} (索引: {square['monitor_index']})")
        print(f"  全局坐标 - 中心: {square['center']}, 左上角: {square['top_left']}")
        print(f"  相对坐标 - 中心: {square['center_relative']}, 左上角: {square['top_left_relative']}")
        print(f"  大小: {square['width']} x {square['height']}")
        print(f"  面积: {square['area']}")
        print()
    
    # 只在第一个显示器上寻找蓝色正方形
    # if len(detector.monitors) > 0:
    #     print("在第一个显示器上寻找蓝色正方形...")
    #     blue_squares = detector.find_colored_squares(
    #         color_bgr=(255, 0, 0),  # 蓝色
    #         min_area=200,
    #         max_area=2000,
    #         monitor_index=0
    #     )
        
    #     print(f"在显示器0上找到 {len(blue_squares)} 个蓝色正方形")
    
    # 根据特定大小寻找绿色正方形
    # print("寻找边长约为50像素的绿色正方形...")
    # green_squares = detector.find_squares_by_size(
    #     color_bgr=(0, 255, 0),  # 绿色
    #     target_size=50,
    #     size_tolerance=10
    # )
    
    # print(f"找到 {len(green_squares)} 个绿色正方形")
    
    # 可视化结果（如果找到了任何正方形）
    all_squares = red_squares #  + blue_squares + green_squares
    if all_squares:
        detector.visualize_results(all_squares, save_path="detection_results")
        
    return all_squares

if __name__ == "__main__":
    all_squares = main()


可用显示器:
  0: \\.\DISPLAY5 [主显示器]
    位置: (0, 0)
    分辨率: 2560 x 1440

  1: \\.\DISPLAY1
    位置: (-2560, -67)
    分辨率: 2133 x 1333

在所有显示器上寻找正方形...
找到 72 个正方形:
正方形 1:
  显示器: \\.\DISPLAY5 (索引: 0)
  全局坐标 - 中心: (2450, 1352), 左上角: (2443, 1346)
  相对坐标 - 中心: (2450, 1352), 左上角: (2443, 1346)
  大小: 14 x 13
  面积: 156.0

正方形 2:
  显示器: \\.\DISPLAY5 (索引: 0)
  全局坐标 - 中心: (1977, 1352), 左上角: (1971, 1346)
  相对坐标 - 中心: (1977, 1352), 左上角: (1971, 1346)
  大小: 12 x 13
  面积: 132.0

正方形 3:
  显示器: \\.\DISPLAY5 (索引: 0)
  全局坐标 - 中心: (2056, 1312), 左上角: (2050, 1306)
  相对坐标 - 中心: (2056, 1312), 左上角: (2050, 1306)
  大小: 13 x 13
  面积: 144.0

正方形 4:
  显示器: \\.\DISPLAY5 (索引: 0)
  全局坐标 - 中心: (1938, 1312), 左上角: (1932, 1306)
  相对坐标 - 中心: (1938, 1312), 左上角: (1932, 1306)
  大小: 12 x 13
  面积: 132.0

正方形 5:
  显示器: \\.\DISPLAY5 (索引: 0)
  全局坐标 - 中心: (875, 1273), 左上角: (869, 1267)
  相对坐标 - 中心: (875, 1273), 左上角: (869, 1267)
  大小: 12 x 13
  面积: 132.0

正方形 6:
  显示器: \\.\DISPLAY5 (索引: 0)
  全局坐标 - 中心: (836, 1273), 左上角: (830, 1267)
  相对坐标 -

In [ ]:
CURRENT_CHARGE = 30
COLOR_BGR      = (123, 123, 123)
MIN_AREA       = 100
MAX_AREA       = 200

detector = MultiMonitorSquareDetector()
    
# 列出所有显示器
detector.list_monitors()

# 在所有显示器上寻找正方形
print("在所有显示器上寻找正方形...")
red_squares = detector.find_colored_squares(
    color_bgr=COLOR_BGR,
    min_area=MIN_AREA,
    max_area=MAX_AREA
)

count = 0
print(f"找到 {len(red_squares)} 个正方形:")
for i, square in enumerate(red_squares):
    print(f"正方形 {i+1}:")
    print(f"  显示器: {square['monitor_name']} (索引: {square['monitor_index']})")
    print(f"  全局坐标 - 中心: {square['center']}, 左上角: {square['top_left']}")
    print(f"  相对坐标 - 中心: {square['center_relative']}, 左上角: {square['top_left_relative']}")
    print(f"  大小: {square['width']} x {square['height']}")
    print(f"  面积: {square['area']}")
    print()
    x, y = square['center']
    pyautogui.moveTo(x, y, duration=random.randint(100, 500) / 1000)
    pyautogui.click(x, y)
    count += 1
    time.sleep(random.randint(100, 500) / 1000)
    if count >= CURRENT_CHARGE:
        break
    

可用显示器:
  0: \\.\DISPLAY5 [主显示器]
    位置: (0, 0)
    分辨率: 2560 x 1440

  1: \\.\DISPLAY1
    位置: (-2560, -67)
    分辨率: 2133 x 1333

在所有显示器上寻找正方形...
找到 74 个正方形:
正方形 1:
  显示器: \\.\DISPLAY5 (索引: 0)
  全局坐标 - 中心: (2514, 1045), 左上角: (2508, 1039)
  相对坐标 - 中心: (2514, 1045), 左上角: (2508, 1039)
  大小: 13 x 13
  面积: 144.0

正方形 2:
  显示器: \\.\DISPLAY5 (索引: 0)
  全局坐标 - 中心: (2199, 1045), 左上角: (2193, 1039)
  相对坐标 - 中心: (2199, 1045), 左上角: (2193, 1039)
  大小: 13 x 13
  面积: 144.0

正方形 3:
  显示器: \\.\DISPLAY5 (索引: 0)
  全局坐标 - 中心: (1727, 1045), 左上角: (1721, 1039)
  相对坐标 - 中心: (1727, 1045), 左上角: (1721, 1039)
  大小: 12 x 13
  面积: 132.0

正方形 4:
  显示器: \\.\DISPLAY5 (索引: 0)
  全局坐标 - 中心: (2514, 1006), 左上角: (2508, 1000)
  相对坐标 - 中心: (2514, 1006), 左上角: (2508, 1000)
  大小: 13 x 13
  面积: 144.0

正方形 5:
  显示器: \\.\DISPLAY5 (索引: 0)
  全局坐标 - 中心: (1806, 1006), 左上角: (1800, 1000)
  相对坐标 - 中心: (1806, 1006), 左上角: (1800, 1000)
  大小: 12 x 13
  面积: 132.0

正方形 6:
  显示器: \\.\DISPLAY5 (索引: 0)
  全局坐标 - 中心: (1688, 1006), 左上角: (1682, 1000)
  